---
title: "Public IP enrichment: worked example"
execute:
  freeze: auto
---

Companion notebook for the site page *Public IP enrichment: what the flags actually mean*. The implementation lives in `openbotrisk.ip_enrichment`; this notebook downloads immutable raw snapshots, measures coverage, and demonstrates the stable output schema.

`True` means matched, `False` means checked and not matched, and `NA` means the source could not be applied. Network access is needed only when this notebook is explicitly executed. Rendering the site uses the outputs stored here.

In [1]:
#| label: setup
import datetime
import json
from ipaddress import ip_network
from pathlib import Path

import pandas as pd

from openbotrisk.ip_enrichment import (
    IntervalSet,
    ListedIPs,
    RouteTable,
    artifact_provenance,
    enrich,
    provider_sets,
    snapshot,
)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO / "data/raw/ip-enrichment" / datetime.date.today().isoformat()
IPTOASN_PATH = REPO / "data/IP/ip2asn-combined.tsv.gz"

In [2]:
#| label: snapshots
# Reuse a dated local snapshot by default; set this explicitly to refresh live data.
REFRESH_SOURCES = False

def cached_or_download(name, url):
    path = RAW / name
    manifest_path = RAW / "manifest.jsonl"
    if path.exists() and not REFRESH_SOURCES:
        records = []
        if manifest_path.exists():
            records = [json.loads(line) for line in manifest_path.read_text().splitlines()]
        prior = next((row for row in reversed(records) if row.get("name") == name), {})
        return artifact_provenance(
            name,
            url,
            path,
            retrieved_at=prior.get("retrieved_at"),
            source_updated_at=prior.get("source_updated_at"),
        )
    return snapshot(name, url, RAW)

SOURCES = {
    "x4bnet_vpn_ipv4.txt": "https://raw.githubusercontent.com/X4BNet/lists_vpn/main/output/vpn/ipv4.txt",
    "x4bnet_datacenter_ipv4.txt": "https://raw.githubusercontent.com/X4BNet/lists_vpn/main/output/datacenter/ipv4.txt",
    "rezmoss_all_providers.json": "https://raw.githubusercontent.com/rezmoss/cloud-provider-ip-addresses/main/all_providers/all_providers.json",
    "bad_asn_list.csv": "https://raw.githubusercontent.com/brianhama/bad-asn-list/master/bad-asn-list.csv",
}
snaps = {name: cached_or_download(name, url) for name, url in SOURCES.items()}

# Default list of addresses observed by Tor's active measurements as exits.
try:
    snaps["tor_bulk_exit.txt"] = cached_or_download(
        "tor_bulk_exit.txt", "https://check.torproject.org/torbulkexitlist"
    )
except OSError as exc:
    print("Tor list unavailable; tor_exit_listed will be NA:", exc)

In [3]:
#| label: load-sources
vpn = IntervalSet.from_file(snaps["x4bnet_vpn_ipv4.txt"].path)
datacentre = IntervalSet.from_file(snaps["x4bnet_datacenter_ipv4.txt"].path)

providers = pd.read_json(snaps["rezmoss_all_providers.json"].path)
cloud = provider_sets(providers)  # Provider-level matches; overlaps are retained.

bad_asn = pd.read_csv(snaps["bad_asn_list.csv"].path).drop_duplicates("ASN")
bad_asns = set(bad_asn["ASN"])

tor = (
    ListedIPs.from_file(snaps["tor_bulk_exit.txt"].path)
    if "tor_bulk_exit.txt" in snaps
    else None
)
route = RouteTable.from_tsv(IPTOASN_PATH) if IPTOASN_PATH.exists() else None
if route is None:
    print("No route table found: route_asn and hosting_asn_listed will be NA.")

## Coverage and dependence

In [4]:
#| label: coverage-table
cloud_any = IntervalSet(
    ip_network(cidr, strict=False)
    for cidr in providers.loc[providers["ip_version"] == "IPv4", "cidr"]
)
sets = {
    "X4BNet VPN": vpn,
    "X4BNet datacentre": datacentre,
    "rezmoss provider union (IPv4)": cloud_any,
}
coverage = pd.DataFrame(
    {name: {"networks": len(value), "addresses": value.addresses} for name, value in sets.items()}
).T
coverage["pct_of_ipv4"] = (coverage["addresses"] / 2**32 * 100).round(2)
coverage

,networks,addresses,pct_of_ipv4
X4BNet VPN,10931,3170072,0.07
X4BNet datacentre,44584,395597872,9.21
rezmoss provider union (IPv4),12842,255279702,5.94


In [5]:
#| label: overlap
overlap = {
    "provider union also in X4BNet datacentre": (
        cloud_any.overlap_addresses(datacentre) / cloud_any.addresses
    ),
    "X4BNet VPN also in X4BNet datacentre": (
        vpn.overlap_addresses(datacentre) / vpn.addresses
    ),
}
pd.Series(overlap).map("{:.1%}".format)

provider union also in X4BNet datacentre    91.7%
X4BNet VPN also in X4BNet datacentre        99.2%
dtype: object

## Enrich some addresses

The demo includes IPv4, IPv6, IPv4-mapped IPv6, malformed input, and null input so the distinction between `False` and `NA` is visible.

In [6]:
#| label: demo-ips
demo = [
    "52.95.110.1",      # AWS range
    "8.8.8.8",          # Google resolver; infrastructure is not an abuse verdict
    "81.2.69.142",      # Ordinary UK address
    "2001:db8::1",      # IPv6 -> NA for current IPv4-only interval sources
    "::ffff:34.82.1.5", # Deliberately normalized to IPv4
    "1.2.3.4:443",      # Port attached -> unparseable -> NA
    "not an ip",
    None,
]

In [7]:
#| label: enrich-demo
out = enrich(
    demo,
    route=route,
    cloud=cloud,
    tor=tor,
    vpn=vpn,
    datacentre=datacentre,
    bad_asns=bad_asns,
)
out

,ip_input,ip_normalized,ip_version,route_asn,asn_name,asn_country,cloud_matches,tor_exit_listed,vpn_listed,datacentre_listed,hosting_asn_listed
0,52.95.110.1,52.95.110.1,4,16509,AMAZON-02,US,"(aws,)",False,False,True,True
1,8.8.8.8,8.8.8.8,4,15169,GOOGLE,US,"(googleservices,)",False,False,True,True
2,81.2.69.142,81.2.69.142,4,20712,AS20712 Andrews & Arnold Ltd,GB,(),False,False,False,False
3,2001:db8::1,2001:db8::1,6,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,::ffff:34.82.1.5,34.82.1.5,4,396982,GOOGLE-CLOUD-PLATFORM,US,"(googlecloud,)",False,False,True,False
5,1.2.3.4:443,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,not an ip,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


`cloud_matches` is a tuple containing every provider match; an empty tuple means checked with no match. `NA` means the source could not be applied.

In [8]:
#| label: na-semantics
# Report this share next to any flag rate.
out.drop(columns=["ip_input"]).isna().mean().rename("NA share").round(2)

ip_normalized         0.38
ip_version            0.38
route_asn             0.50
asn_name              0.50
asn_country           0.50
cloud_matches         0.50
tor_exit_listed       0.50
vpn_listed            0.50
datacentre_listed     0.50
hosting_asn_listed    0.50
Name: NA share, dtype: float64

## Provenance

In [9]:
#| label: provenance
source_records = [
    {
        "file": item.name,
        "retrieved_at": item.retrieved_at,
        "file_mtime": item.file_mtime,
        "source_updated_at": item.source_updated_at,
        "sha256": item.sha256,
        "url": item.url,
    }
    for item in snaps.values()
]
if IPTOASN_PATH.exists():
    route_artifact = artifact_provenance(
        IPTOASN_PATH.name,
        "https://iptoasn.com/data/ip2asn-combined.tsv.gz",
        IPTOASN_PATH,
    )
    source_records.append(
        {
            "file": route_artifact.name,
            "retrieved_at": route_artifact.retrieved_at,
            "file_mtime": route_artifact.file_mtime,
            "source_updated_at": route_artifact.source_updated_at,
            "sha256": route_artifact.sha256,
            "url": route_artifact.url,
        }
    )
pd.DataFrame(source_records)

,file,retrieved_at,file_mtime,source_updated_at,sha256,url
0,x4bnet_vpn_ipv4.txt,2026-10-25T10:54:43+00:00,2026-10-25T10:54:43+00:00,None,d5555f06a151141d08d79bbbcb96150f979860b48d8bfd...,https://raw.githubusercontent.com/X4BNet/lists...
1,x4bnet_datacenter_ipv4.txt,2026-10-25T10:54:43+00:00,2026-10-25T10:54:43+00:00,None,380eccc1e05d05b3535dff6c380a0fb9d8bd6978018c97...,https://raw.githubusercontent.com/X4BNet/lists...
2,rezmoss_all_providers.json,2026-10-25T10:54:51+00:00,2026-10-25T10:54:51+00:00,None,fa5c03243d283b27af15bf7cad73f42cf768e21f9c0654...,https://raw.githubusercontent.com/rezmoss/clou...
3,bad_asn_list.csv,2026-10-25T10:54:51+00:00,2026-10-25T10:54:51+00:00,None,0af8e58dc07ba070a10c9083a1a9f5c9fc76be35fab32b...,https://raw.githubusercontent.com/brianhama/ba...
4,tor_bulk_exit.txt,2026-10-25T10:54:51+00:00,2026-10-25T10:54:51+00:00,None,6756af70704aa18c358c48563b54ec27c962b86f0dc2b5...,https://check.torproject.org/torbulkexitlist
5,ip2asn-combined.tsv.gz,None,2026-10-24T12:38:06+00:00,None,7336abc7611351b0261023c6268293bc9d4a02fb32a3ea...,https://iptoasn.com/data/ip2asn-combined.tsv.gz
